In [1]:
import pandas as pd
import numpy as np

df_temp1 = pd.read_csv('../temp_25_26.csv')
df_temp2 = pd.read_csv('../temp_24_25.csv')
df_temp3 = pd.read_csv('../temp_23_24.csv')
df_temp4 = pd.read_csv('../temp_22_23.csv')
df_temp5 = pd.read_csv('../temp_21_22.csv')
df_temp6 = pd.read_csv('../temp_20_21.csv') 

df_master = pd.concat([df_temp1, df_temp2, df_temp3, df_temp4, df_temp5, df_temp6], ignore_index=True)

columnas_clave = [
    'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 
    'HS', 'AS', 'HST', 'AST', 
    'HC', 'AC'
]

df_def = df_master[columnas_clave].copy()

df_def['Date'] = pd.to_datetime(df_def['Date'], format='%d/%m/%Y', errors='coerce')
df_def = df_def.sort_values(by='Date').reset_index(drop=True)

print(f"Total de partidos cargados: {df_def.shape[0]}")
df_def.head()

Total de partidos cargados: 2189


,Date,HomeTeam,AwayTeam,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HC,AC
0,2020-09-19,Fiorentina,Torino,1,0,0,0,17,6,8,3,4,1
1,2020-09-19,Verona,Roma,0,0,0,0,9,15,7,7,0,6
2,2020-09-20,Juventus,Sampdoria,3,0,1,0,14,10,9,2,6,4
3,2020-09-20,Parma,Napoli,0,2,0,0,4,12,2,7,4,2
4,2020-09-20,Genoa,Crotone,4,1,3,1,10,11,8,5,6,4


In [2]:
# 1. Rachas de GOLES de los últimos 5 partidos (A favor y en contra)
df_def['Racha_Local_GF'] = df_def.groupby('HomeTeam')['FTHG'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Local_GC'] = df_def.groupby('HomeTeam')['FTAG'].transform(lambda x: x.rolling(window=5, closed='left').mean())

df_def['Racha_Visita_GF'] = df_def.groupby('AwayTeam')['FTAG'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_GC'] = df_def.groupby('AwayTeam')['FTHG'].transform(lambda x: x.rolling(window=5, closed='left').mean())

# 2. Rachas OFENSIVAS (Tiros a puerta y Córners a favor)
df_def['Racha_Local_TirosPuerta_Favor'] = df_def.groupby('HomeTeam')['HST'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_TirosPuerta_Favor'] = df_def.groupby('AwayTeam')['AST'].transform(lambda x: x.rolling(window=5, closed='left').mean())

df_def['Racha_Local_Corners_Favor'] = df_def.groupby('HomeTeam')['HC'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_Corners_Favor'] = df_def.groupby('AwayTeam')['AC'].transform(lambda x: x.rolling(window=5, closed='left').mean())

# 3. Rachas DEFENSIVAS (Tiros a puerta y Córners CONCEDIDOS) - ¡TU NUEVA IDEA!
# Lo que concede el local, son los tiros (AST) y córners (AC) de la visita.
df_def['Racha_Local_TirosPuerta_Contra'] = df_def.groupby('HomeTeam')['AST'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Local_Corners_Contra'] = df_def.groupby('HomeTeam')['AC'].transform(lambda x: x.rolling(window=5, closed='left').mean())

# Lo que concede la visita, son los tiros (HST) y córners (HC) del local.
df_def['Racha_Visita_TirosPuerta_Contra'] = df_def.groupby('AwayTeam')['HST'].transform(lambda x: x.rolling(window=5, closed='left').mean())
df_def['Racha_Visita_Corners_Contra'] = df_def.groupby('AwayTeam')['HC'].transform(lambda x: x.rolling(window=5, closed='left').mean())

# 4. Historial Directo (H2H) - Promedio de goles cuando se enfrentan
df_def['H2H_GF_Local'] = df_def.groupby(['HomeTeam', 'AwayTeam'])['FTHG'].transform(lambda x: x.shift(1).expanding().mean())
df_def['H2H_GF_Visita'] = df_def.groupby(['HomeTeam', 'AwayTeam'])['FTAG'].transform(lambda x: x.shift(1).expanding().mean())

promedio_gf_liga = df_def['FTHG'].mean()
promedio_gc_liga = df_def['FTAG'].mean()
df_def['H2H_GF_Local'] = df_def['H2H_GF_Local'].fillna(promedio_gf_liga)
df_def['H2H_GF_Visita'] = df_def['H2H_GF_Visita'].fillna(promedio_gc_liga)

df_def = df_def.dropna().reset_index(drop=True)

print(f"Datos para entrenar: {df_def.shape[0]} partidos.")
df_def[['Date', 'HomeTeam', 'AwayTeam', 'Racha_Local_TirosPuerta_Contra', 'Racha_Visita_TirosPuerta_Contra']].tail()

Datos para entrenar: 2001 partidos.


,Date,HomeTeam,AwayTeam,Racha_Local_TirosPuerta_Contra,Racha_Visita_TirosPuerta_Contra
1996,2026-03-15,Verona,Genoa,4.4,5.6
1997,2026-03-15,Sassuolo,Bologna,6.6,3.8
1998,2026-03-15,Como,Roma,2.4,4.2
1999,2026-03-15,Lazio,Milan,4.8,4.4
2000,2026-03-15,Pisa,Cagliari,3.8,4.8


In [3]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

columnas_features = [
    'Racha_Local_GF', 'Racha_Local_GC',
    'Racha_Visita_GF', 'Racha_Visita_GC',
    'Racha_Local_TirosPuerta_Favor', 'Racha_Visita_TirosPuerta_Favor',
    'Racha_Local_Corners_Favor', 'Racha_Visita_Corners_Favor',
    'Racha_Local_TirosPuerta_Contra', 'Racha_Visita_TirosPuerta_Contra', 
    'Racha_Local_Corners_Contra', 'Racha_Visita_Corners_Contra',
    'H2H_GF_Local', 'H2H_GF_Visita'
]

X = df_def[columnas_features]

y_local = np.clip(df_def['FTHG'], 0, 3) 
y_visita = np.clip(df_def['FTAG'], 0, 3)

X_train, X_test, y_train_local, y_test_local = train_test_split(X, y_local, test_size=0.2, random_state=42)
_, _, y_train_visita, y_test_visita = train_test_split(X, y_visita, test_size=0.2, random_state=42)

modelo_goles_local = xgb.XGBClassifier(objective='multi:softprob', num_class=4, n_estimators=100, learning_rate=0.05, random_state=42)
modelo_goles_visita = xgb.XGBClassifier(objective='multi:softprob', num_class=4, n_estimators=100, learning_rate=0.05, random_state=42)

# 5. Entrenar
print("Entrenando Clasificador para goles del Local...")
modelo_goles_local.fit(X_train, y_train_local)

print("Entrenando Clasificador para goles del Visitante...")
modelo_goles_visita.fit(X_train, y_train_visita)

predicciones_local = modelo_goles_local.predict(X_test)
predicciones_visita = modelo_goles_visita.predict(X_test)

acc_local = accuracy_score(y_test_local, predicciones_local)
acc_visita = accuracy_score(y_test_visita, predicciones_visita)

print("\n" + "="*50)
print("PRECISIÓN EN ADIVINAR EL NÚMERO EXACTO DE GOLES")
print("="*50)
print(f"Local: {acc_local * 100:.2f}% de veces adivinó los goles exactos.")
print(f"Visita: {acc_visita * 100:.2f}% de veces adivinó los goles exactos.")

import pandas as pd
ejemplo_df = pd.DataFrame({
    'Goles_Reales_Local': y_test_local.values[:10],
    'Predicción_Local': predicciones_local[:10],
    'Goles_Reales_Visita': y_test_visita.values[:10],
    'Predicción_Visita': predicciones_visita[:10]
})

print("\nEjemplo de las primeras 10 predicciones:")
print(ejemplo_df)

Entrenando Clasificador para goles del Local...
Entrenando Clasificador para goles del Visitante...

PRECISIÓN EN ADIVINAR EL NÚMERO EXACTO DE GOLES
Local: 30.42% de veces adivinó los goles exactos.
Visita: 35.16% de veces adivinó los goles exactos.

Ejemplo de las primeras 10 predicciones:
   Goles_Reales_Local  Predicción_Local  Goles_Reales_Visita  \
0                   1                 1                    2   
1                   2                 0                    1   
2                   0                 1                    2   
3                   0                 0                    3   
4                   2                 1                    2   
5                   3                 0                    2   
6                   2                 1                    1   
7                   0                 0                    2   
8                   3                 1                    0   
9                   2                 3                    2   

   

In [4]:
import pandas as pd

importancias_local = modelo_goles_local.feature_importances_
importancias_visita = modelo_goles_visita.feature_importances_

df_importancias = pd.DataFrame({
    'Estadística (Variable)': columnas_features,
    'Peso en Modelo LOCAL (%)': importancias_local * 100,
    'Peso en Modelo VISITA (%)': importancias_visita * 100
})

df_importancias = df_importancias.sort_values(by='Peso en Modelo LOCAL (%)', ascending=False).reset_index(drop=True)

print("¿QUÉ MIRA EL MODELO PARA PREDECIR LOS GOLES?\n")
display(df_importancias.round(2))

print("\nInterpretación:")
print("- Si el porcentaje es alto, significa que el modelo depende mucho de esa estadística.")
print("- Si el porcentaje es bajo, significa que esa estadística casi no afectó su decisión final.")

¿QUÉ MIRA EL MODELO PARA PREDECIR LOS GOLES?



,Estadística (Variable),Peso en Modelo LOCAL (%),Peso en Modelo VISITA (%)
0,Racha_Local_TirosPuerta_Favor,10.04,6.78
1,Racha_Visita_TirosPuerta_Contra,7.83,7.20
2,H2H_GF_Local,7.37,7.73
3,Racha_Visita_TirosPuerta_Favor,7.32,6.83
4,Racha_Visita_GC,7.27,7.19
5,Racha_Visita_Corners_Contra,7.27,7.21
6,H2H_GF_Visita,7.14,7.81
7,Racha_Visita_GF,7.00,7.39
8,Racha_Local_Corners_Contra,6.98,7.34
9,Racha_Local_GF,6.77,7.37



Interpretación:
- Si el porcentaje es alto, significa que el modelo depende mucho de esa estadística.
- Si el porcentaje es bajo, significa que esa estadística casi no afectó su decisión final.


In [5]:
import requests
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') 

match_name_mapping = {
"Juventus FC": "Juventus",
    "AC Milan": "Milan",
    "FC Internazionale Milano": "Inter",
    "SSC Napoli": "Napoli",
    "AS Roma": "Roma",
    "SS Lazio": "Lazio",
    "ACF Fiorentina": "Fiorentina",
    "Atalanta BC": "Atalanta",
    "Bologna FC 1909": "Bologna",
    "Torino FC": "Torino",
    "Genoa CFC": "Genoa",
    "Hellas Verona FC": "Verona",
    "Empoli FC": "Empoli",
    "US Lecce": "Lecce",
    "Udinese Calcio": "Udinese",
    "Cagliari Calcio": "Cagliari",
    "AC Monza": "Monza",
    "Parma Calcio 1913": "Parma",
    "Venezia FC": "Venezia",
    "Como 1907": "Como",
    # Equipos de temporadas anteriores recientes por si están en tu historial:
    "US Sassuolo Calcio": "Sassuolo",
    "Frosinone Calcio": "Frosinone",
    "US Salernitana 1919": "Salernitana",
    "Spezia Calcio": "Spezia",
    "UC Sampdoria": "Sampdoria",
    "US Cremonese": "Cremonese",
    "AC Pisa 1909": "Pisa"
}

promedio_gf_liga = df_master['FTHG'].mean()
promedio_gc_liga = df_master['FTAG'].mean()

def predecir_partido_tabla(fecha, local, visita, modelo_local, modelo_visita, df_base):
    
    df_jugados = df_base.dropna(subset=['FTHG', 'FTAG']).copy()
    
    df_jugados['Date'] = pd.to_datetime(df_jugados['Date'], dayfirst=True, errors='coerce')
    
    df_local = df_jugados[df_jugados['HomeTeam'] == local].sort_values('Date').tail(5)
    df_visita = df_jugados[df_jugados['AwayTeam'] == visita].sort_values('Date').tail(5)
    
    if len(df_local) < 5 or len(df_visita) < 5:
        return None 
          
    racha_local_gf = df_local['FTHG'].mean()
    racha_local_gc = df_local['FTAG'].mean()
    racha_local_tp_favor = df_local['HST'].mean()
    racha_local_cor_favor = df_local['HC'].mean()
    racha_local_tp_contra = df_local['AST'].mean() 
    racha_local_cor_contra = df_local['AC'].mean()
    
    racha_visita_gf = df_visita['FTAG'].mean()
    racha_visita_gc = df_visita['FTHG'].mean()
    racha_visita_tp_favor = df_visita['AST'].mean()
    racha_visita_cor_favor = df_visita['AC'].mean()
    racha_visita_tp_contra = df_visita['HST'].mean() 
    racha_visita_cor_contra = df_visita['HC'].mean()
    
    h2h_historico = df_base[(df_base['HomeTeam'] == local) & (df_base['AwayTeam'] == visita)]
    if len(h2h_historico) > 0:
        h2h_gf_local = h2h_historico['FTHG'].mean()
        h2h_gf_visita = h2h_historico['FTAG'].mean()
    else:
        h2h_gf_local = promedio_gf_liga
        h2h_gf_visita = promedio_gc_liga
        
    datos_hoy = pd.DataFrame([[
        racha_local_gf, racha_local_gc, racha_visita_gf, racha_visita_gc,
        racha_local_tp_favor, racha_visita_tp_favor, racha_local_cor_favor, racha_visita_cor_favor,
        racha_local_tp_contra, racha_visita_tp_contra, racha_local_cor_contra, racha_visita_cor_contra,
        h2h_gf_local, h2h_gf_visita
    ]], columns=columnas_features) 
    
    pred_entero_local = int(modelo_local.predict(datos_hoy)[0])
    pred_entero_visita = int(modelo_visita.predict(datos_hoy)[0])
    
    probs_local = modelo_local.predict_proba(datos_hoy)[0]
    probs_visita = modelo_visita.predict_proba(datos_hoy)[0]
    
    xg_local = sum(i * prob for i, prob in enumerate(probs_local))
    xg_visita = sum(i * prob for i, prob in enumerate(probs_visita))
    
    return {
        "Fecha": fecha,
        "Local": local,
        "Visita": visita,
        "xG L": round(xg_local, 2),
        "xG V": round(xg_visita, 2),
        "MARCADOR": f"{pred_entero_local} - {pred_entero_visita}",
        "GFL": round(racha_local_gf, 1),
        "GCL": round(racha_local_gc, 1),
        "TPLF": round(racha_local_tp_favor, 1),
        "TPLC": round(racha_local_tp_contra, 1),
        "GFV": round(racha_visita_gf, 1),
        "GCV": round(racha_visita_gc, 1),
        "TPVF": round(racha_visita_tp_favor, 1),
        "TPVC": round(racha_visita_tp_contra, 1)
    }

API_KEY = "65104bce29144179b5b85d53cf565eaa" 
URL_PL_MATCHES = "https://api.football-data.org/v4/competitions/SA/matches"

headers = {"X-Auth-Token": API_KEY}
params = {"status": "SCHEDULED"}

respuesta = requests.get(URL_PL_MATCHES, headers=headers, params=params)

if respuesta.status_code == 200:
    lista_partidos = respuesta.json()["matches"][:10]
    resultados = []
    
    for partido in lista_partidos:
        fecha_utc = partido["utcDate"][:10]
        nombre_api_local = partido["homeTeam"]["name"]
        nombre_api_visitante = partido["awayTeam"]["name"]
        
        equipo_modelo_local = match_name_mapping.get(nombre_api_local, nombre_api_local)
        equipo_modelo_visitante = match_name_mapping.get(nombre_api_visitante, nombre_api_visitante)
        
        try:
            fila_resultado = predecir_partido_tabla(fecha_utc, equipo_modelo_local, equipo_modelo_visitante, modelo_goles_local, modelo_goles_visita, df_master)
            if fila_resultado:
                resultados.append(fila_resultado)
        except Exception as e:
            pass 
            
    # Mostrar la tabla final
    df_resultados = pd.DataFrame(resultados)
    
    print("\nPREDICCIONES GENERADAS CON ÉXITO:\n")
    display(df_resultados)

else:
    print(f"Error al conectar con la API. Código: {respuesta.status_code}")


PREDICCIONES GENERADAS CON ÉXITO:



,Fecha,Local,Visita,xG L,xG V,MARCADOR,GFL,GCL,TPLF,TPLC,GFV,GCV,TPVF,TPVC
0,2026-03-20,Cagliari,Napoli,0.67,1.02,0 - 0,1.2,0.8,2.6,3.4,1.6,2.0,3.6,4.4
1,2026-03-20,Genoa,Udinese,1.27,0.76,1 - 0,2.6,1.2,5.2,3.0,1.6,1.4,3.4,4.6
2,2026-03-21,Parma,Cremonese,1.41,0.98,1 - 1,0.8,1.6,3.6,5.0,0.4,2.6,2.6,7.0
3,2026-03-21,Milan,Torino,1.87,0.84,2 - 1,0.8,0.6,5.2,2.2,0.6,3.0,3.0,6.2
4,2026-03-21,Juventus,Sassuolo,1.98,1.02,3 - 1,2.8,0.8,7.0,1.8,1.2,1.4,3.8,4.6
5,2026-03-22,Como,Pisa,1.85,0.67,3 - 0,2.4,0.8,6.6,1.8,0.8,2.6,3.2,5.6
6,2026-03-22,Atalanta,Verona,1.84,1.01,3 - 0,2.4,0.8,6.8,2.6,0.6,2.0,2.8,4.4
7,2026-03-22,Bologna,Lazio,1.08,0.93,1 - 1,0.6,1.6,3.0,3.2,0.6,0.8,2.2,3.8
8,2026-03-22,Roma,Lecce,1.67,0.75,1 - 1,2.2,0.8,5.6,2.2,0.8,1.4,2.0,4.4
9,2026-03-22,Fiorentina,Inter,1.01,1.35,0 - 0,1.0,1.0,4.2,2.8,2.0,0.2,6.2,1.6
